# 3c · The ARIA chat app + structured output — SOLUTION

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath("../.."))
from shared.llm import chat

ARIA_SYSTEM = (
    "You are ARIA, the calm, concise assistant for the Orbital Mars colony. "
    "Never invent data; ask for confirmation before any physical action."
)

def new_conversation():
    return [{"role": "system", "content": ARIA_SYSTEM}]

def say(history, user_text):
    history.append({"role": "user", "content": user_text})
    reply = chat(messages=history)
    history.append({"role": "assistant", "content": reply})
    return history, reply

history = new_conversation()
history, reply = say(history, "CO2 in Module B just hit 1400 ppm.")
print("ARIA:", reply)
history, reply = say(history, "What was the module I just mentioned?")
print("ARIA:", reply)

In [ ]:
ASSESS_SYSTEM = (
    "You are ARIA. Assess the reading and reply with ONLY a JSON object, no prose, "
    'with keys: severity ("nominal"|"amber"|"red"), action (short string), rationale (short string).'
)

def assess(reading_text):
    raw = chat(reading_text, system=ASSESS_SYSTEM, temperature=0)
    raw = raw.strip().strip("`")
    if raw.startswith("json"):
        raw = raw[4:].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # one retry with a firmer instruction
        raw = chat(reading_text + "\nReturn ONLY valid JSON.", system=ASSESS_SYSTEM, temperature=0)
        return json.loads(raw.strip().strip("`").removeprefix("json").strip())

print(assess("O2 is 18.7% and CO2 is 1600 ppm in Module B"))